In [ ]:
 # Dataset con problemas:

In [1]:
import pandas as pd
import numpy as np

# Crear datos de ejemplo con problemas
ventas = pd.DataFrame({
    'producto': ['A', 'B', None, 'A', 'C'],
    'precio': [100, None, 150, 100, 200],
    'cantidad': [1, 2, None, 1, 3],
    'fecha': ['2024-01-01', None, '2024-01-03', '2024-01-01', 'invalid']
})

print("Datos originales:")
print(ventas)
print(f"Valores faltantes por columna:\n{ventas.isnull().sum()}")

Datos originales:
  producto  precio  cantidad       fecha
0        A   100.0       1.0  2024-01-01
1        B     NaN       2.0        None
2     None   150.0       NaN  2024-01-03
3        A   100.0       1.0  2024-01-01
4        C   200.0       3.0     invalid
Valores faltantes por columna:
producto    1
precio      1
cantidad    1
fecha       1
dtype: int64


In [ ]:
 # Limpiar datos:

In [2]:
def limpiar_datos_ventas(df):
    df_limpio = df.copy()
    
    # 1. Eliminar duplicados
    df_limpio = df_limpio.drop_duplicates()
    
    # 2. Imputar valores faltantes
    df_limpio['precio'] = df_limpio['precio'].fillna(df_limpio['precio'].median())
    df_limpio['cantidad'] = df_limpio['cantidad'].fillna(1)  # Asumir cantidad mínima
    
    # 3. Eliminar filas con producto faltante
    df_limpio = df_limpio.dropna(subset=['producto'])
    
    # 4. Corregir fechas inválidas
    df_limpio['fecha'] = pd.to_datetime(df_limpio['fecha'], errors='coerce')
    df_limpio = df_limpio.dropna(subset=['fecha'])
    
    # 5. Calcular total
    df_limpio['total'] = df_limpio['precio'] * df_limpio['cantidad']
    
    return df_limpio

ventas_limpias = limpiar_datos_ventas(ventas)
print("\nDatos limpios:")
print(ventas_limpias)
print(f"\nRegistros finales: {len(ventas_limpias)}")


Datos limpios:
  producto  precio  cantidad      fecha  total
0        A   100.0       1.0 2024-01-01  100.0

Registros finales: 1


In [ ]:
# Validar datos limpios:

In [3]:
def validar_ventas_limpias(df):
    validaciones = {
        'sin_faltantes': df.isnull().sum().sum() == 0,
        'precios_positivos': (df['precio'] > 0).all(),
        'cantidades_positivas': (df['cantidad'] > 0).all(),
        'fechas_validas': pd.api.types.is_datetime64_any_dtype(df['fecha']),
        'total_correcto': np.allclose(df['total'], df['precio'] * df['cantidad'])
    }
    
    print("Validaciones:")
    for check, passed in validaciones.items():
        status = "✅" if passed else "❌"
        print(f"  {status} {check}")
    
    return all(validaciones.values())

es_valido = validar_ventas_limpias(ventas_limpias)
print(f"\nDataset válido: {es_valido}")

Validaciones:
  ✅ sin_faltantes
  ✅ precios_positivos
  ✅ cantidades_positivas
  ✅ fechas_validas
  ✅ total_correcto

Dataset válido: True


In [ ]:
""" ¿Cuándo eliminar datos faltantes vs imputarlos?
Eliminar datos faltantes (drop)
Funciona mejor cuando:
• La proporción de datos faltantes es baja
Si menos del 3–5% de las filas están incompletas, eliminarlas suele ser más seguro que imputar.
• Los valores faltantes no son sistemáticos
Si el patrón es aleatorio, eliminar no introduce sesgo.
• La fila completa pierde sentido sin ese dato
Ejemplo: registros de ventas sin fecha o sin ID de producto.
• El volumen de datos es grande
En datasets masivos, perder un pequeño porcentaje no afecta la representatividad.
• El campo es clave o no imputable
Ejemplo: RUT, ID de transacción, timestamp exacto.

Imputar datos faltantes
Conviene cuando:
• El dato es recuperable o estimable
Ejemplo: ingresos, edad, precios, métricas numéricas.
• Eliminar filas generaría sesgo
Si los faltantes no son aleatorios, imputar preserva la estructura del dataset.
• El dataset es pequeño
Perder filas puede destruir la capacidad analítica.
•El campo es relevante para el modelo
Ejemplo: variables predictoras en modelos ML.
• Hay relaciones fuertes entre variables
Permite imputaciones más inteligentes (regresión, KNN, modelos bayesianos).

Métodos comunes de imputación
• Numéricos: media, mediana, regresión, KNN, interpolación temporal.
• Categóricos: moda, categoría “desconocido”, imputación por árbol de decisión.
• Series de tiempo: forward fill, backward fill, interpolación lineal.

¿Qué validaciones son más importantes según el tipo de dato?

1. Datos numéricos
Validaciones clave:
• Rangos válidos
Ej: precio ≥ 0, edad entre 0 y 120.
• Distribución esperada
Detectar outliers con IQR, Z-score o MAD.
• Coherencia entre variables
• Unidades consistentes
Ej: CLP vs USD, kWh vs Wh.

2. Datos categóricos
Validaciones clave:
• Dominio permitido
Ej: método de pago ∈ {“efectivo”, “tarjeta”, “transferencia”}.
• Normalización de texto
Mayúsculas, tildes, espacios, variantes (“Santiago”, “Stgo”, “SCL”).
• Cardinalidad razonable
Detectar categorías raras o errores tipográficos.

3. Fechas y tiempos
Validaciones clave:
• Formato correcto
ISO 8601 o estándar definido.
• Secuencia lógica
Ej: fecha de entrega ≥ fecha de compra.
• Rangos válidos
Ej: no fechas futuras en transacciones históricas.
• 	Timezone consistente

4. Identificadores (IDs, claves, RUT, UUID)
Validaciones clave:
• Unicidad
No duplicados en claves primarias.
• Integridad referencial
FK debe existir en la tabla padre.
• Formato válido
Ej: validación de dígito verificador del RUT.

5. Series de tiempo
Validaciones clave:
• Frecuencia consistente
Horaria, diaria, semanal.
• No saltos inesperados
Días faltantes, duplicados, timestamps fuera de orden.

6. Datos derivados o calculados
Validaciones clave:
• Reproducibilidad
El cálculo debe ser determinístico.
• Coherencia matemática
Ej: porcentajes que suman 100%.
• Dependencias claras
Documentar qué columnas alimentan el cálculo.

>>>>>>>>>> Resumen <<<<<<<<<<

• Eliminar cuando el dato es clave, no imputable o el porcentaje es bajo.
• Imputar cuando eliminar introduce sesgo o reduce demasiado la muestra.
• Validar según el tipo de dato: rangos, dominios, secuencias, integridad y coherencia.

"""